# Dependency injection

**Objective:** Replace infrastructure without changing endpoint or business logic.

## Simple version

In [ ]:
# Depends asks FastAPI to supply the value instead of creating it in the route.
from fastapi import Depends, FastAPI


app = FastAPI()


def get_version() -> str:
    return "v1"


@app.get("/version")
async def version(value: str = Depends(get_version)) -> dict:
    return {"version": value}


print(get_version())

## Polished version

In [ ]:
# The route depends on a repository contract, so storage is replaceable.
from typing import Annotated, Protocol

import httpx
from fastapi import Depends, FastAPI, Request


class TaskRepository(Protocol):
    async def count(self) -> int: ...


class MemoryTaskRepository:
    def __init__(self, count: int = 0) -> None:
        self.value = count

    async def count(self) -> int:
        return self.value


def get_repository(request: Request) -> TaskRepository:
    return request.app.state.tasks


Repository = Annotated[TaskRepository, Depends(get_repository)]


def create_app(repository: TaskRepository) -> FastAPI:
    app = FastAPI()
    # The composition root installs the chosen adapter on application state.
    app.state.tasks = repository

    @app.get("/tasks/count")
    async def task_count(tasks: Repository) -> dict:
        return {"count": await tasks.count()}

    return app


# Inject an in-memory adapter so this HTTP test needs no database.
app = create_app(MemoryTaskRepository(count=2))
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    response = await client.get("/tasks/count")

print(response.json())